# 06 · Mapa de Whitechapel — tipografía y papel

**V2:** descarta redibujar cartografía. Retoca el mapa REAL sin mover ni un marcador; no promete eliminar el solapamiento de letras (requiere edición vectorial aparte).

**Objetivo:** corregir el solapamiento de nombres del `room_map.jpg` original y crear un mapa narrativo limpio para inspección, conservando **anclas de interacción (700,280,r175)** y las marcas del diseño actual. No utiliza modelos generativos: mapas, letras y pistas dibujadas por IA difusiva suelen deformarse.

El mapa nuevo es una **reconstrucción artística del mapa del videojuego**, **no un plano histórico cartográficamente certificado**. Verificar todos los nombres y posiciones con la documentación del juego antes de aprobar. Las cruces y el círculo están ubicados de forma aproximada según la ilustración existente. Comparar a resolución nativa 1280×720 y dentro del farol del juego. No cambiar las frases de las pistas ni el código de inspección.

Solo CPU; no consume GPU de Colab.

In [ ]:
%pip -q install 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
out=Path('/content/horrorplay_map_candidates');out.mkdir(parents=True,exist_ok=True)
old,sha=source('web/assets/closeups/room_map.jpg');assert old.size==(1280,720)
print('Original sha256:',sha)
display(old.resize((960,540)))

In [ ]:
# V2: prohibido redibujar calles, cruces, emplazamientos o texto.
# El mapa completo ya existe. Retocamos únicamente luminancia/contraste global
# para evitar perder su topología al inventar otra cartografía.
import numpy as np
from PIL import ImageEnhance, ImageFilter
W,H=1280,720

def grade_map(im,delta):
    src=np.asarray(im,dtype=np.int16)
    lum=src.mean(axis=2)
    # Levantar áreas oscuras del papel, respetar trazos entintados.
    lift=np.where(lum>120,delta,0).astype(np.int16)
    corrected=np.clip(src+lift[...,None],0,255).astype(np.uint8)
    return Image.fromarray(corrected,'RGB')
versions=[grade_map(old,6),grade_map(old,12)]
for name,im in zip(['A_fiel_mas_claro','B_fiel_legible'],versions):
    im.save(out/f'room_map_{name}.png')
    im.save(out/f'room_map_{name}.jpg',quality=91,optimize=True)
    # Ningún píxel se mueve. Toda modificación de color <=12.
    dif=np.abs(np.asarray(im,dtype=np.int16)-np.asarray(old,dtype=np.int16))
    assert dif.max()<=12
sheet=compare([old,*versions],(640,360),['Actual','A +6 (mismo mapa)','B +12 (mismo mapa)'])
sheet.save(out/'AB_map_640.png');display(sheet)
print('Topología/hotspots conservados. Texto original superpuesto NO corregido; necesita retoque tipográfico manual.')


## Control del hotspot

El punto inspeccionable sigue siendo el círculo `(700,280,r175)` de `room.js`. Su posición no se cambia. Se dibuja como guía **sobre una copia de revisión**, no sobre el JPG final. Verifica que la cartela y rosa de los vientos no formen parte del círculo. Revisa también la posición y ortografía histórica de los rótulos antes de aceptar.

In [ ]:
review=versions[0].copy().convert('RGBA')
layer=Image.new('RGBA',review.size,(0,0,0,0))
rd=ImageDraw.Draw(layer)
rd.ellipse((700-175,280-175,700+175,280+175),outline=(200,15,18,240),width=5)
review=Image.alpha_composite(review,layer)
display(review.resize((960,540)))
review.save(out/'QA_hotspot_MAP_NO_INTEGRAR.png')
finish(out,{'source_sha256':sha,'hotspot_center':[700,280],'hotspot_radius':175,
  'warning':'V2 conserva exactamente geometría y cruces; NO resuelve solapamientos del texto original. Revisar legibilidad antes de integrar.'},[])